if it helpful for you please upvote the notebook.
# Breast Cancer Classification using PyTorch

In this notebook, we will build a complete **Breast Cancer Classification Model** using PyTorch while understanding the core components of a deep learning workflow.

### Topics Covered

#### 1. Autograd

PyTorch's automatic differentiation engine that computes gradients automatically during backpropagation. It eliminates the need to manually calculate derivatives and enables efficient neural network training.

#### 2. nn.Module

The base class for all neural networks in PyTorch. It provides a structured way to define layers, parameters, and the forward pass of a model.

#### 3. Custom Dataset Class

A user-defined class inherited from `torch.utils.data.Dataset` that specifies how data is stored, accessed, and returned to the model.

#### 4. DataLoader

A utility that loads data in mini-batches, supports shuffling, and efficiently feeds data to the model during training and evaluation.

#### 5. Training Loop

The iterative process where the model performs forward propagation, computes loss, calculates gradients, and updates parameters over multiple epochs.

#### 6. Loss Function

A function that measures the difference between the model's predictions and the actual target values. The objective of training is to minimize this value.

#### 7. Optimizer

An algorithm responsible for updating model parameters using gradients computed by Autograd. Examples include SGD and Adam.

#### 8. Model Evaluation

The process of measuring the model's performance on unseen data using metrics such as accuracy, precision, recall, and F1-score.

---

## Project Objective

The goal of this project is to build a binary classification model that predicts whether a breast tumor is:

* **Malignant (M)** → Cancerous Tumor
* **Benign (B)** → Non-Cancerous Tumor

using the Breast Cancer Wisconsin dataset and PyTorch.

---

## Workflow Overview

```text
Dataset
   ↓
Data Preprocessing
   ↓
Custom Dataset
   ↓
DataLoader
   ↓
Neural Network (nn.Module)
   ↓
Forward Pass
   ↓
Loss Calculation
   ↓
Autograd (Gradient Computation)
   ↓
Optimizer (Parameter Update)
   ↓
Training Loop
   ↓
Model Evaluation
```

### Expected Learning Outcomes

By the end of this notebook, you will understand:

* How PyTorch tensors work
* How Autograd computes gradients
* How to create custom Dataset classes
* How DataLoader creates mini-batches
* How to build neural networks using `nn.Module`
* How forward and backward propagation work
* How optimizers update model weights
* How to train a model from scratch
* How to evaluate a classification model

This project covers the fundamental PyTorch concepts required for most Machine Learning and Deep Learning projects.


In [253]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [254]:
df = pd.read_csv('/kaggle/input/datasets/yasserh/breast-cancer-dataset/breast-cancer.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [255]:
df.tail()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
564,926424,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,926682,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,926954,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,927241,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400
568,92751,B,7.76,24.54,47.92,181.0,0.05263,0.04362,0.00000,0.00000,...,9.456,30.37,59.16,268.6,0.08996,0.06444,0.0000,0.0000,0.2871,0.07039


In [256]:
df['diagnosis'].unique()

array(['M', 'B'], dtype=object)

In [257]:
df['diagnosis'].value_counts()

diagnosis
B    357
M    212
Name: count, dtype: int64

In [258]:
df.shape

(569, 32)

In [259]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

# NAN values

In [260]:
df.isna().any().any()


np.False_

In [261]:
df.isna().sum()

id                         0
diagnosis                  0
radius_mean                0
texture_mean               0
perimeter_mean             0
area_mean                  0
smoothness_mean            0
compactness_mean           0
concavity_mean             0
concave points_mean        0
symmetry_mean              0
fractal_dimension_mean     0
radius_se                  0
texture_se                 0
perimeter_se               0
area_se                    0
smoothness_se              0
compactness_se             0
concavity_se               0
concave points_se          0
symmetry_se                0
fractal_dimension_se       0
radius_worst               0
texture_worst              0
perimeter_worst            0
area_worst                 0
smoothness_worst           0
compactness_worst          0
concavity_worst            0
concave points_worst       0
symmetry_worst             0
fractal_dimension_worst    0
dtype: int64

In [262]:
df.describe()

,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
count,5.690000e+02,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,3.037183e+07,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,1.250206e+08,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,8.670000e+03,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,8.692180e+05,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,9.060240e+05,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,8.813129e+06,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,9.113205e+08,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


# Duplicate Check

In [263]:
df.duplicated().sum()

np.int64(0)

In [264]:
df.dtypes

id                           int64
diagnosis                   object
radius_mean                float64
texture_mean               float64
perimeter_mean             float64
area_mean                  float64
smoothness_mean            float64
compactness_mean           float64
concavity_mean             float64
concave points_mean        float64
symmetry_mean              float64
fractal_dimension_mean     float64
radius_se                  float64
texture_se                 float64
perimeter_se               float64
area_se                    float64
smoothness_se              float64
compactness_se             float64
concavity_se               float64
concave points_se          float64
symmetry_se                float64
fractal_dimension_se       float64
radius_worst               float64
texture_worst              float64
perimeter_worst            float64
area_worst                 float64
smoothness_worst           float64
compactness_worst          float64
concavity_worst     

In [265]:
df.columns

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst'],
      dtype='object')

# Drop Unnecessary Columns 

In [266]:
df = df.drop(columns = ['id'])

In [267]:
# df['diagnosis'] = df['diagnosis'].map({
#     'M':1,
#     'B':0
# })


In [268]:
df

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


# Corrilation

In [269]:
df['diagnosis'] = df['diagnosis'].map({
    'M':1,
    'B':0
})
corr = df.corr()

In [270]:
corr

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
diagnosis,1.000000,0.730029,0.415185,0.742636,0.708984,0.358560,0.596534,0.696360,0.776614,0.330499,...,0.776454,0.456903,0.782914,0.733825,0.421465,0.590998,0.659610,0.793566,0.416294,0.323872
radius_mean,0.730029,1.000000,0.323782,0.997855,0.987357,0.170581,0.506124,0.676764,0.822529,0.147741,...,0.969539,0.297008,0.965137,0.941082,0.119616,0.413463,0.526911,0.744214,0.163953,0.007066
texture_mean,0.415185,0.323782,1.000000,0.329533,0.321086,-0.023389,0.236702,0.302418,0.293464,0.071401,...,0.352573,0.912045,0.358040,0.343546,0.077503,0.277830,0.301025,0.295316,0.105008,0.119205
perimeter_mean,0.742636,0.997855,0.329533,1.000000,0.986507,0.207278,0.556936,0.716136,0.850977,0.183027,...,0.969476,0.303038,0.970387,0.941550,0.150549,0.455774,0.563879,0.771241,0.189115,0.051019
area_mean,0.708984,0.987357,0.321086,0.986507,1.000000,0.177028,0.498502,0.685983,0.823269,0.151293,...,0.962746,0.287489,0.959120,0.959213,0.123523,0.390410,0.512606,0.722017,0.143570,0.003738
smoothness_mean,0.358560,0.170581,-0.023389,0.207278,0.177028,1.000000,0.659123,0.521984,0.553695,0.557775,...,0.213120,0.036072,0.238853,0.206718,0.805324,0.472468,0.434926,0.503053,0.394309,0.499316
compactness_mean,0.596534,0.506124,0.236702,0.556936,0.498502,0.659123,1.000000,0.883121,0.831135,0.602641,...,0.535315,0.248133,0.590210,0.509604,0.565541,0.865809,0.816275,0.815573,0.510223,0.687382
concavity_mean,0.696360,0.676764,0.302418,0.716136,0.685983,0.521984,0.883121,1.000000,0.921391,0.500667,...,0.688236,0.299879,0.729565,0.675987,0.448822,0.754968,0.884103,0.861323,0.409464,0.514930
concave points_mean,0.776614,0.822529,0.293464,0.850977,0.823269,0.553695,0.831135,0.921391,1.000000,0.462497,...,0.830318,0.292752,0.855923,0.809630,0.452753,0.667454,0.752399,0.910155,0.375744,0.368661
symmetry_mean,0.330499,0.147741,0.071401,0.183027,0.151293,0.557775,0.602641,0.500667,0.462497,1.000000,...,0.185728,0.090651,0.219169,0.177193,0.426675,0.473200,0.433721,0.430297,0.699826,0.438413


 | VIF Value    | Meaning                                       |
| ------------ | --------------------------------------------- |
| VIF = 1      | No multicollinearity                          |
| 1 < VIF < 5  | Low multicollinearity                         |
| 5 ≤ VIF < 10 | Moderate multicollinearity                    |
| VIF ≥ 10     | High multicollinearity (remove/check feature) |


In [271]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df.drop('diagnosis', axis=1)

vif = []

for i in range(X.shape[1]):
    vif.append(
        variance_inflation_factor(X.values, i)
    )

pd.DataFrame({
    'Feature': X.columns,
    'VIF': vif
}).sort_values('VIF', ascending=False)

,Feature,VIF
0,radius_mean,63306.172036
2,perimeter_mean,58123.586079
20,radius_worst,9674.742602
22,perimeter_worst,4487.781270
3,area_mean,1287.262339
23,area_worst,1138.759252
9,fractal_dimension_mean,629.679874
29,fractal_dimension_worst,423.396723
4,smoothness_mean,393.398166
24,smoothness_worst,375.597155


 Breast Cancer dataset me many features are naturally correlated because they represent related geometric measurements. Removing all high-VIF features would cause significant information loss. Instead, I removed only redundant highly correlated features 

 I decided to remove those columns which carry same information .

these three groups have same info .... but in Deep learning it not necessary to remove these type of column because NEural network handle them by itself.
```
radius_mean
perimeter_mean
area_mean

radius_worst
perimeter_worst
area_worst

radius_se
perimeter_se
area_se```


MultiColinarity means the other column how much variation define of other column


In [272]:
cols_to_drop = [
    'perimeter_mean',
    'area_mean',

    'perimeter_worst',
    'area_worst',

    'perimeter_se',
    'area_se'
]

df = df.drop(columns=cols_to_drop)

# Split Features and Target

In [273]:
X = df.drop(columns=['diagnosis'])

y = df['diagnosis']

# Train-Test Split

In [274]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

# Encoding

In [275]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# Feature Scaling

In [276]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

# Numpy arrays to PyTorch tensors

In [277]:

X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [278]:
from torch.utils.data import Dataset,DataLoader

# Dataset Class
The CustomDataset class is used to organize and manage data in a format that PyTorch can easily work with during training. It inherits from Dataset and defines three important methods: __init__, __len__, and __getitem__. The __init__ method stores the input features and labels, the __len__ method returns the total number of samples in the dataset, and the __getitem__ method retrieves a single feature-label pair based on a given index. This allows PyTorch's DataLoader to efficiently load data in batches, shuffle samples, and iterate through the dataset during training. By creating a custom dataset class, we can easily use our own data with PyTorch's training pipeline while keeping the code clean, reusable, and scalable for larger projects.

In [279]:
class CustomDataset(Dataset):
    def __init__(self,feature,labels): # labels is actually target column
        
        self.feature = feature
        self.labels = labels

    def __len__(self):
        
        return len(self.feature)
        
    def __getitem__(self,index):
        
        return self.feature[index],self.labels[index]
        
        

In [280]:
# objects
train_dataset = CustomDataset(X_train_tensor,y_train_tensor)
test_dataset = CustomDataset(X_test_tensor,y_test_tensor)

In [281]:
len(train_dataset)

455

# DataLoader Class
The DataLoader is a PyTorch utility that takes a dataset (such as a custom dataset class) and provides an efficient way to load data during training and evaluation. Instead of feeding the entire dataset to the model at once, the DataLoader automatically divides the data into smaller batches, optionally shuffles the samples, and returns one batch at a time. This makes training faster, more memory-efficient, and easier to manage, especially when working with large datasets. It also supports features such as parallel data loading through multiple workers and automatic batching, allowing the training loop to focus only on forward propagation, loss calculation, and optimization while the DataLoader handles data preparation behind the scenes.

Parameters of DataLoader Class

| Parameter            | Purpose                                 | Default         | Kab Use Hota Hai                         |
| -------------------- | --------------------------------------- | --------------- | ---------------------------------------- |
| `dataset`            | Data source                             | Required        | Dataset object pass karte hain           |
| `batch_size`         | Kitne samples ek batch me               | `1`             | Training ko mini-batches me divide karna |
| `shuffle`            | Data ko random order me karna           | `False`         | Training me almost hamesha `True`        |
| `sampler`            | Custom sampling strategy                | `None`          | RandomSampler, WeightedRandomSampler     |
| `batch_sampler`      | Direct batches generate karta hai       | `None`          | Advanced custom batching                 |
| `num_workers`        | Parallel workers for loading            | `0`             | Large datasets/images                    |
| `collate_fn`         | Samples ko batch me combine karta hai   | Default collate | NLP, variable-length sequences           |
| `pin_memory`         | CPU→GPU transfer fast karta hai         | `False`         | GPU training me                          |
| `drop_last`          | Last incomplete batch ko drop karta hai | `False`         | BatchNorm training me useful             |
| `timeout`            | Worker response wait time               | `0`             | Rarely used                              |
| `worker_init_fn`     | Har worker start par function call      | `None`          | Random seed setting                      |
| `persistent_workers` | Workers ko alive rakhta hai             | `False`         | Multiple epochs me speedup               |
| `prefetch_factor`    | Worker pehle se batches prepare kare    | `2`             | Large datasets                           |
| `generator`          | Custom random generator                 | `None`          | Reproducibility                          |


In [282]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

# Defining the model

In [283]:
import torch.nn as nn

In [284]:
class MySimpleNN(nn.Module):
    def __init__(self,n_features):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(n_features,128),
            nn.ReLU(),
            
            nn.Linear(128,64),
            nn.ReLU(),
            
            nn.Linear(64,32),
            nn.ReLU(),
            
            nn.Linear(32,1),
            nn.Sigmoid()
        )
    def forward(self,features):
        return self.network(features)
    

In [285]:
X_train_tensor.shape[1]

24

In [286]:
len(train_dataset)

455

In [287]:
X_train_tensor.shape

torch.Size([455, 24])

In [288]:
# pipline 

learning_rate = 0.1
epochs = 25

model = MySimpleNN(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate)

loss_function = nn.BCELoss()


In [289]:

# define loop
for epoch in range(epochs):

  for batch_features, batch_labels in train_loader:

    # forward pass ..# batch size opr 32 rkha is liye yaha 32 rows jaygi hr bar
    y_pred = model(batch_features) 

    # loss calculate
    loss = loss_function(y_pred, batch_labels.view(-1,1))

    # clear gradients
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # parameters update
    optimizer.step()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.5959475636482239
Epoch: 2, Loss: 0.41233521699905396
Epoch: 3, Loss: 0.3204062879085541
Epoch: 4, Loss: 0.09680620580911636
Epoch: 5, Loss: 0.0856170579791069
Epoch: 6, Loss: 0.07883372157812119
Epoch: 7, Loss: 0.09718881547451019
Epoch: 8, Loss: 0.04004523530602455
Epoch: 9, Loss: 0.014008636586368084
Epoch: 10, Loss: 0.04535108059644699
Epoch: 11, Loss: 0.0328015498816967
Epoch: 12, Loss: 0.02146671526134014
Epoch: 13, Loss: 0.10057691484689713
Epoch: 14, Loss: 0.016812071204185486
Epoch: 15, Loss: 0.001643051509745419
Epoch: 16, Loss: 0.06249335780739784
Epoch: 17, Loss: 0.007384355179965496
Epoch: 18, Loss: 0.002376521471887827
Epoch: 19, Loss: 0.001163773238658905
Epoch: 20, Loss: 0.01080903597176075
Epoch: 21, Loss: 0.11749054491519928
Epoch: 22, Loss: 0.014004410244524479
Epoch: 23, Loss: 0.00018519445438869298
Epoch: 24, Loss: 0.013025445863604546
Epoch: 25, Loss: 0.024958107620477676


In [290]:
# Model evaluation using test_loader
model.eval()  # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # Forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary predictions

        # Calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')


Accuracy: 0.9688
